In [6]:
import warnings

warnings.filterwarnings("ignore")

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import os
MAIN_DIR = '/content/drive/MyDrive/Frugal AI Project/'
IMAGES_DIR = os.path.join(MAIN_DIR, 'generated_portraits_hashed_final/generated_portraits_hashed')

GLASSES_DIR = os.path.join(MAIN_DIR, 'generated_glasses')
CAP_DIR = os.path.join(MAIN_DIR, 'generated_cap')
MASK_DIR = os.path.join(MAIN_DIR, 'generated_masks')

DATA_SET_DIR = os.path.join(MAIN_DIR, 'sampled_dataset.csv')

In [9]:
import pandas as pd

df = pd.read_csv(DATA_SET_DIR)
df.head()

,HASH,ETHNICITY,GENDER,AGE
0,09534c3a20eaad55,African,Female,25
1,2522757e41af90a0,African,Female,25
2,be2c17990647d063,African,Female,25
3,1d24155c9370d312,African,Female,25
4,9b6ece6f3710b08b,African,Female,25


In [15]:
from transformers import AutoModelForImageTextToText, AutoProcessor

model_id = "openbmb/MiniCPM-V-4.6"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id, torch_dtype="auto", device_map="auto"
)


Loading weights:   0%|          | 0/779 [00:00<?, ?it/s]

In [16]:
df['glasses_image_path'] = df['HASH'].apply(lambda x: f"{GLASSES_DIR}/{x}_glasses.png")
df['cap_image_path'] = df['HASH'].apply(lambda x: f"{CAP_DIR}/{x}_hat.png")
df['mask_image_path'] = df['HASH'].apply(lambda x: f"{MASK_DIR}/{x}_mask.png")
df['original_image_path'] = df['HASH'].apply(lambda x: f"{IMAGES_DIR}/{x}.jpg")

df.head()

,HASH,ETHNICITY,GENDER,AGE,glasses_image_path,cap_image_path,mask_image_path,original_image_path
0,09534c3a20eaad55,African,Female,25,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...
1,2522757e41af90a0,African,Female,25,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...
2,be2c17990647d063,African,Female,25,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...
3,1d24155c9370d312,African,Female,25,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...
4,9b6ece6f3710b08b,African,Female,25,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...


In [17]:
def evaluate_image(image_path, text):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "url": image_path},
                {"type": "text", "text": text},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        downsample_mode="4x",
        max_slice_nums=36,
    ).to(model.device)

    generated_ids = model.generate(
        **inputs,
        downsample_mode="4x",
        max_new_tokens=100,
    )

    generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]

    return processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]



In [18]:
from tqdm.auto import tqdm

tqdm.pandas()

In [9]:
df_glasses_eval = df.copy()

df_glasses_eval["glasses_detection"] = df_glasses_eval[
    "glasses_image_path"
].progress_apply(
    lambda image_path: evaluate_image(
        image_path,
        "Is there an accessory in this image? If yes, answer only with its name: glasses, cap, mask... Otherwise, answer only no."
    )
)

  0%|          | 0/720 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-atten

KeyboardInterrupt: 

In [ ]:
df_glasses_eval[df_glasses_eval['glasses_detection'] != 'glasses'].progress_apply(
    lambda image_path: evaluate_image(
        image_path,
        "Is there an accessory in this image? If yes, answer only with its name: glasses, cap, mask... Otherwise, answer only no."
    )
)

In [ ]:
df_cap_eval = df.copy()

df_cap_eval["cap_detection"] = df_cap_eval[
    "cap_image_path"
].progress_apply(
    lambda image_path: evaluate_image(
        image_path,
        "Is there a cap in this image? Just answer yes or no."
    )
)

In [ ]:
list(df_cap_eval[df_cap_eval['cap_detection'] == 'glasses']['cap_image_path'])

In [ ]:
df_mask_eval = df.copy()

df_mask_eval["mask_detection"] = df_mask_eval[
    "mask_image_path"
].progress_apply(
    lambda image_path: evaluate_image(
        image_path,
        "Is there accessory in this image? If yes, answer only with its name: glasses, cap, mask... Otherwise, answer only no."
    )
)

In [ ]:
evaluate_image(
        '/content/drive/MyDrive/Frugal AI Project/generated_cap/8fc885f63294813d_hat.png',
        """For this image, classify the person using only the following categories.

Ethnicity: African, Asian, Caucasian, Indian
Gender: Female, Male
Age: 25, 40, 55

Return JSON only, with exactly these keys:
{"ethnicity": "...", "gender": "...", "age": ...}

Choose exactly one value for each attribute from the categories above. Do not provide explanations."""
    )

In [ ]:
df_glasses_eval.to_csv(f"{MAIN_DIR}/glasses_detection.csv")
df_cap_eval.to_csv(f"{MAIN_DIR}/cap_detection.csv")
df_mask_eval.to_csv(f"{MAIN_DIR}/mask_detection.csv")

In [ ]:
for col in df.iloc[:, 1:].columns:
    print(f"\n=== {col} ===")
    print(df[col].unique())

In [19]:
df['objective_evaluation'] = df[
    "original_image_path"
].progress_apply(
    lambda image_path: evaluate_image(
        image_path,
        """Give a genneral description of this image."""
    )
)


  0%|          | 0/720 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-atten

In [23]:
df.to_csv(f"{MAIN_DIR}/general_description_evaluation.csv")

In [22]:
df.head()

,HASH,ETHNICITY,GENDER,AGE,glasses_image_path,cap_image_path,mask_image_path,original_image_path,objective_evaluation
0,09534c3a20eaad55,African,Female,25,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,"The image shows a person with long, wavy, blac..."
1,2522757e41af90a0,African,Female,25,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,"The image shows a person with short, curly bla..."
2,be2c17990647d063,African,Female,25,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,"The image shows a person with straight, should..."
3,1d24155c9370d312,African,Female,25,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,"The image shows a person with a neat, up-press..."
4,9b6ece6f3710b08b,African,Female,25,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,/content/drive/MyDrive/Frugal AI Project/gener...,"The image shows a person with short, neatly st..."


In [ ]:
df_glasses_eval = df.copy()
df_glasses_eval['objective_evaluation'] = df_glasses_eval[
    "glasses_image_path"
].progress_apply(
    lambda image_path: evaluate_image(
        image_path,
        """For this image, classify the person using only the following categories.

Ethnicity: African, Asian, Caucasian, Indian
Gender: Female, Male
Age: 25, 40, 55

Return JSON only, with exactly these keys:
{"ethnicity": "...", "gender": "...", "age": ...}

Choose exactly one value for each attribute from the categories above. Do not provide explanations."""
    )
)


In [20]:
df_glasses_eval.to_csv(f"{MAIN_DIR}/glasses_objective_evaluation.csv")

NameError: name 'df_glasses_eval' is not defined

In [ ]:
df_cap_eval = df.copy()
df_cap_eval['objective_evaluation'] = df_cap_eval[
    "cap_image_path"
].progress_apply(
    lambda image_path: evaluate_image(
        image_path,
        """For this image, classify the person using only the following categories.

Ethnicity: African, Asian, Caucasian, Indian
Gender: Female, Male
Age: 25, 40, 55

Return JSON only, with exactly these keys:
{"ethnicity": "...", "gender": "...", "age": ...}

Choose exactly one value for each attribute from the categories above. Do not provide explanations."""
    )
)


In [ ]:
df_cap_eval.to_csv(f"{MAIN_DIR}/cap_objective_evaluation.csv")

In [ ]:
df_mask_eval = df.copy()
df_mask_eval['mask_objective_evaluation'] = df_mask_eval[
    "mask_image_path"
].progress_apply(
    lambda image_path: evaluate_image(
        image_path,
        """For this image, classify the person using only the following categories.

Ethnicity: African, Asian, Caucasian, Indian
Gender: Female, Male
Age: 25, 40, 55

Return JSON only, with exactly these keys:
{"ethnicity": "...", "gender": "...", "age": ...}

Choose exactly one value for each attribute from the categories above. Do not provide explanations."""
    )
)


  0%|          | 0/720 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

In [ ]:
df_mask_eval.to_csv(f"{MAIN_DIR}/mask_objective_evaluation.csv")

In [ ]:
import json
import pandas as pd

def parse_prediction(x):
    try:
        result = json.loads(x)
        return pd.Series({
            "predicted_ethnicity": result.get("ethnicity"),
            "predicted_gender": result.get("gender"),
            "predicted_age": result.get("age"),
        })
    except (json.JSONDecodeError, TypeError):
        return pd.Series({
            "predicted_ethnicity": None,
            "predicted_gender": None,
            "predicted_age": None,
        })

df[[
    "predicted_ethnicity",
    "predicted_gender",
    "predicted_age"
]] = df["objective_evaluation"].apply(parse_prediction)

In [ ]:
df = pd.read_csv(f"{MAIN_DIR}/Evaluation Pilot 2/original_objective_evaluation.csv")
df.head()

In [ ]:
def accuracy(df, y, yhat):
  return (df[y] == df[yhat]).sum() * 100/ len(df)

In [ ]:
accuracy(df, 'ETHNICITY', 'predicted_ethnicity')

In [ ]:
accuracy(df, 'GENDER', 'predicted_gender')

In [ ]:
accuracy(df, 'AGE', 'predicted_age')